<a href="https://colab.research.google.com/github/davidekim/WRAPs/blob/main/symm_wraps.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**General pipeline for creating WRAPed symmetric homo-oligomers**

The code to generate block adjacencies interactively originated from [diffusion_foldcond](https://colab.research.google.com/github/sokrypton/ColabDesign/blob/v1.1.1/rf/examples/diffusion_foldcond.ipynb). We thank Sergey Ovchinnikov for developing such a useful tool. We also like to thank Will Sheffler for developing the code used to prepare symmetric inputs for RFdiffusion.

In [ ]:
#@title **Install PyMOL for symmetry utility functions** (rerun after restarting session)

!pip install pymol-open-source-whl

import sys
sys.path.append('/usr/local/lib/python3.12/dist-packages')

In [ ]:
#@title **Setup RFdiffusion and pyrosetta** (~5-10min)
%%time
import os, time
import sys
import subprocess

def run_cmd(cmd):
  process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, shell=True, text=True)
  for line in iter(process.stdout.readline, ''):
    sys.stdout.write(line)
    sys.stdout.flush()
  process.stdout.close()
  process.wait()

if not os.path.isdir("RFdiffusion"):
  print("installing RFdiffusion...")
  os.system("git clone https://github.com/RosettaCommons/RFdiffusion.git")
  # install dependencies
  os.system("pip install jedi omegaconf hydra-core icecream pyrsistent pynvml decorator")
  os.system("pip install git+https://github.com/NVIDIA/dllogger#egg=dllogger")
  os.system("pip install --no-dependencies dgl -f https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html")
  os.system("pip install --no-dependencies e3nn==0.5.5 opt_einsum_fx")
  os.system("cd RFdiffusion/env/SE3Transformer; pip install .")
  os.system("pip install biopython==1.81")
  os.system("pip install -U dm-haiku")
  os.system("pip install ml-collections")
  os.system('pip install --upgrade "jax[cuda12_pip]<0.6.0" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html')
  os.system("cd RFdiffusion; pip install -e .")
  # install PyRosetta
  os.system("pip install pyrosetta --find-links https://west.rosettacommons.org/pyrosetta/quarterly/release")
  os.system("pip install py3Dmol")
print()

os.environ["DGLBACKEND"] = "pytorch"
#os.environ["HYDRA_FULL_ERROR"] = '1'

In [ ]:
#@title **Install RFdiffusion weights** (~1-5min)
%%time
if not os.path.isdir("RFdiffusion/models"):
  print("installing RFdiffusion weights...")
  # Only install what is used in this notebook
  os.system("cd RFdiffusion; mkdir models && cd models")
  #os.system("cd RFdiffusion/models; wget http://files.ipd.uw.edu/pub/RFdiffusion/6f5902ac237024bdd0c176cb93063dc4/Base_ckpt.pt")
  #os.system("cd RFdiffusion/models; wget http://files.ipd.uw.edu/pub/RFdiffusion/e29311f6f1bf1af907f9ef9f44b8328b/Complex_base_ckpt.pt")
  os.system("cd RFdiffusion/models; wget http://files.ipd.uw.edu/pub/RFdiffusion/60f09a193fb5e5ccdc4980417708dbab/Complex_Fold_base_ckpt.pt")
  #os.system("cd RFdiffusion/models; wget http://files.ipd.uw.edu/pub/RFdiffusion/74f51cfb8b440f50d70878e05361d8f0/InpaintSeq_ckpt.pt")
  #os.system("cd RFdiffusion/models; wget http://files.ipd.uw.edu/pub/RFdiffusion/76d00716416567174cdb7ca96e208296/InpaintSeq_Fold_ckpt.pt")
  #os.system("cd RFdiffusion/models; wget http://files.ipd.uw.edu/pub/RFdiffusion/5532d2e1f3a4738decd58b19d633b3c3/ActiveSite_ckpt.pt")
  #os.system("cd RFdiffusion/models; wget http://files.ipd.uw.edu/pub/RFdiffusion/12fc204edeae5b57713c5ad7dcb97d39/Base_epoch8_ckpt.pt")

In [ ]:
#@title **Get ProteinMPNN git repo**
%%time
import os, time
if not os.path.isdir("ProteinMPNN"):
  run_cmd("git clone https://github.com/dauparas/ProteinMPNN.git")

In [ ]:
#@title **Download a homo-oligomeric transmembrane target to WRAP**
import os, sys
import py3Dmol
import ipywidgets as widgets
from ipywidgets import interact, Layout
from IPython.display import display, clear_output
from google.colab import files

#@markdown Leave blank to upload your own PDB. Must be a homo-oligomeric structure.

#@markdown Hover over structure for residue position labels to help select ranges for the next step.

def read_pdb_atom(l):
  chain = l[20:22].strip()
  atype = l[11:17].strip()
  name3 = l[17:20].strip()
  resnum = int(l[22:26].strip())
  x = float(l[30:38])
  y = float(l[38:46])
  z = float(l[46:54])
  return chain, atype, name3, resnum, x, y, z

longer_names = {'ALA': 'A', 'ARG': 'R', 'ASN': 'N', 'ASP': 'D',
              'CYS': 'C', 'GLU': 'E', 'GLN': 'Q', 'GLY': 'G',
              'HIS': 'H', 'ILE': 'I', 'LEU': 'L', 'LYS': 'K',
              'MET': 'M', 'PHE': 'F', 'PRO': 'P', 'SER': 'S',
              'THR': 'T', 'TRP': 'W', 'TYR': 'Y', 'VAL': 'V' }

target_to_wrap = "7ahl" #@param {type:"string"}

if os.path.exists("input_full.pdb"):
  os.system("rm input_full.pdb")

input_pdb_str = ""
if len(target_to_wrap) == 4:
  if not os.path.isfile(f"{target_to_wrap}.pdb1"):
    os.system(f"wget -qnc https://files.rcsb.org/download/{target_to_wrap}.pdb1.gz")
    os.system(f"gunzip {target_to_wrap}.pdb1.gz")
  with open(f"{target_to_wrap}.pdb1") as f:
    for l in f:
        input_pdb_str += l
  with open("input_full.pdb", "w") as out: out.write(input_pdb_str)
elif len(target_to_wrap.strip()) == 0:
  uploads = files.upload()
  if uploads:
    target_to_wrap = list(uploads.keys())[0].split('.pdb')[0].split('/')[-1].split()[0]
    input_pdb_str = uploads[list(uploads.keys())[0]]
    with open("input_full.pdb", "wb") as out: out.write(input_pdb_str)
else:
  raise Exception("pdb must be a 4 character PDB ID")

if os.path.exists("input_full.pdb"):
  clear_output(wait=True)
  view = py3Dmol.view(width=800, height=600)

  pdb_data = ""
  with open("input_full.pdb", "r") as f:
    pdb_data = f.read()
  chains = {}
  for l in pdb_data.split('\n'):
    if l.startswith("ATOM"):
      chain, atype, name3, resnum, x, y, z = read_pdb_atom(l)
      if atype != 'CA': continue
      if not chain in chains:
        chains[chain] = longer_names[name3]
      else:
        chains[chain] += longer_names[name3]
  view.addModel(pdb_data, 'pdb')
  view.setStyle({'cartoon': {'colorscheme': 'chain'}})
  view.setHoverable({}, True,
      '''function(atom,viewer,event,container) {
          if(!atom.label) {
              atom.label = viewer.addLabel(atom.resn + ":" + atom.resi,
                  {position: atom, backgroundColor: 'black', fontColor:'white'});
          }
      }''',
      '''function(atom,viewer) {
          if(atom.label) {
              viewer.removeLabel(atom.label);
              delete atom.label;
          }
      }'''
  )
  view.zoomTo()
  view.show()
else:
  print()
  print("Please provide a PDB before continuing.")

In [ ]:
#@title **Trim asymmetric unit** (make the target more manageable)
import os, sys
import py3Dmol
import ipywidgets as widgets
import re
from ipywidgets import interact, Layout
from IPython.display import display, clear_output

#@markdown Residue range(s) to trim out (\<start>-\<stop> [A:\<start>-\<stop>])

#@markdown Add chain ID followed by : to trim a specific chain.
ranges = "1-109, 150-300" #@param {type:"string"}
trim_ranges = []
for trim in re.split(r'[, ]+',ranges):
  start_stop = trim.split('-')
  chainstr = ''
  try:
    chainid = start_stop[0].split(':')
    if len(chainid) > 1:
      chainstr = chainid[0]
      start_stop[0] = chainid[1]
    start = int(start_stop[0])
    stop = int(start_stop[1])
  except:
    raise Exception("Format in ranges is incorrect.")
  trim_ranges.append({'chain':chainstr,'start':start,'stop':stop})
trimmed_pdb_str = ""
with open("input_full.pdb") as f:
  prevresnum = -999999
  seqresnum = 0
  prevchain = ''
  for l in f:
    if l.startswith("ATOM"):
      chain, atype, name3, resnum, x, y, z = read_pdb_atom(l)
      do_continue = False
      for trim in trim_ranges:
        if (len(trim['chain']) == 0 or trim['chain'] == chain) and resnum >= trim['start'] and resnum <= trim['stop']:
          do_continue = True
          break
      if do_continue: continue
      if chain != prevchain:
        seqresnum = 0
      if resnum != prevresnum:
        seqresnum += 1
      trimmed_pdb_str += l[0:22] + f'{seqresnum:4d}' + l[26:]
      prevresnum = resnum
      prevchain = chain
trimmed_pdb = 'input.pdb'
nf = open(trimmed_pdb, 'w')
nf.write(trimmed_pdb_str)
nf.close()

clear_output(wait=True)
view = py3Dmol.view(width=800, height=600)
with open(trimmed_pdb, "r") as f:
  pdb_data = f.read()

chains = {}
chain_resnums = {}
for l in pdb_data.split('\n'):
  if l.startswith("ATOM"):
    chain, atype, name3, resnum, x, y, z = read_pdb_atom(l)
    if atype != 'CA': continue
    if not chain in chains:
      chains[chain] = longer_names[name3]
    else:
      chains[chain] += longer_names[name3]
    if not chain in chain_resnums:
      chain_resnums[chain] = [ resnum ]
    else:
      chain_resnums[chain].append(resnum)
asymmetric_units = len(chains)
if asymmetric_units < 2:
  raise Exception("There are less than 2 asymmetric units.")
symmetry = f"c{asymmetric_units}"
print(f"{symmetry}")
prevcseq = ""
isSymm = True
for c in chains:
  print(f'{c} {chains[c]}')
  if len(prevcseq) and chains[c] != prevcseq:
    isSymm = False
  prevcseq = chains[c]
if not isSymm:
  raise Exception("Sequence mismatch in symmetric chains.")
chainlen = len(prevcseq)
view.addModel(pdb_data, 'pdb')
view.setStyle({'cartoon': {'colorscheme': 'chain'}})
view.setHoverable({}, True,
    '''function(atom,viewer,event,container) {
        if(!atom.label) {
            atom.label = viewer.addLabel(atom.resn + ":" + atom.resi,
                {position: atom, backgroundColor: 'black', fontColor:'white'});
        }
    }''',
    '''function(atom,viewer) {
        if(atom.label) {
            viewer.removeLabel(atom.label);
            delete atom.label;
        }
    }'''
)
view.zoomTo()
view.show()


In [ ]:
#@title **Align symmetry axis to Z-axis** (for RFdiffusion with symmetry)

if not os.path.exists("sheffler_pymol_symm_utils"):
  os.system("wget https://files.ipd.uw.edu/pub/WRAPs/sheffler_pymol_symm_utils.tar.gz")
  os.system("tar -zxvf sheffler_pymol_symm_utils.tar.gz")
  os.system("touch sheffler_pymol_symm_utils")
sys.path.append('./')
import xyzMath
import xyzGeom
import pymol_util
import sym_util
import symgen
import pymol
from pymol import cmd
import warnings
warnings.filterwarnings('ignore')

centeredz = "input_Z.pdb"
symm = f"C{asymmetric_units}"
pymol.cmd.delete("all")
pymol.importing.load("input.pdb","symm")
chainsstr = ''.join(chains.keys())
sym_util.aligncx(sele='symm', nfold=asymmetric_units, chains=chainsstr)
pymol.cmd.translate([0,0,pymol_util.com("symm")[1]], selection="symm", camera=0)
pymol.cmd.save(centeredz, "symm")
if os.path.exists("input_Z.pdb"):
  os.system("mv input_Z.pdb input.pdb")

In [ ]:
#@title **Initialize secondary structure adjacency matrix with custom WRAP SS blocks**

#@markdown **Add SS blocks to N and/or C terminus for custom WRAPs**

#@markdown Example: H16 L4 H16 L4 H16 (3 16 residue helices connected by 4 residue loops)
N_term_SS_blocks = "H25 L4 H25" #@param {type:"string"}

C_term_SS_blocks = "" #@param {type:"string"}

def ss_blocks(ss_str):
  pos_str = ""
  ss = []
  prevss = ''
  ss_str = ss_str.replace(' ','')
  for c in ss_str:
    if c == 'E' or c == 'L' or c == 'H':
      if len(pos_str) > 0:
        ss.append([prevss,int(pos_str)])
      pos_str = ""
      prevss = c
    else:
      pos_str += c
  if len(pos_str) > 0:
    ss.append([prevss,int(pos_str)])
  return ss

# get target SS blocks
import pyrosetta
import pyrosetta.rosetta.core.import_pose as import_pr_pose
from pyrosetta import *
from pyrosetta.rosetta import *
from pyrosetta.rosetta.core import *
pyrosetta.init(" -mute all ")

def ss_Type_lengths(pose):
  DSSP = pyrosetta.rosetta.protocols.moves.DsspMover()
  DSSP.apply(pose)
  DSSP_pose_string = []
  SS_type_lengths = []
  current_SS = ''
  current_chain = ''
  helix_counter = 0
  Loop_counter = 0
  Sheet_counter = 0
  for z in range(1, len(pose.sequence())+1):
    chainid = pose.chain(z)
    if current_SS == pose.secstruct(z) and current_chain == chainid:
      SS_type_lengths[len(SS_type_lengths)-1][1] = SS_type_lengths[len(SS_type_lengths)-1][1] +1
    if current_SS != pose.secstruct(z) or current_chain != chainid:
      SS_type_lengths.append([pose.secstruct(z),1,chainid])
      if pose.secstruct(z) == "H":
        helix_counter = helix_counter+1
      if pose.secstruct(z) == "L":
        Loop_counter = Loop_counter+1
      if pose.secstruct(z) == "E":
        Sheet_counter = Sheet_counter+1
      current_SS = pose.secstruct(z)
      current_chain = pose.chain(z)
      DSSP_pose_string.append(pose.secstruct(z))
  return SS_type_lengths

pose = import_pr_pose.pose_from_file(trimmed_pdb)

# get chain plus wrap length
cwraplen = 0
for ssblock in ss_blocks(C_term_SS_blocks):
  cwraplen += ssblock[1]
nwraplen = 0
for ssblock in ss_blocks(N_term_SS_blocks):
  nwraplen += ssblock[1]
chain_plus_wrap_len = nwraplen + chainlen + cwraplen
ss_final_chain = []
prevchain = None
for ssblock_ in ss_Type_lengths(pose):
  if ssblock_[2] != prevchain:
    if prevchain:
      break # Just do first asymmetric chain
    for ssblock in ss_blocks(N_term_SS_blocks):
      ssblock.append(ssblock_[2])
      ssblock.append('w')
      ss_final_chain.append(ssblock)
  ssblock_.append('t')
  ss_final_chain.append(ssblock_)
  prevchain = ssblock_[2]
for ssblock in ss_blocks(C_term_SS_blocks):
  ssblock.append(prevchain)
  ssblock.append('w')
  ss_final_chain.append(ssblock)
ss_final = []
print("[SS, len, chain number, wrap (w) or target (t)]")
for cid in range(1,len(chains)+1):
  for ss in ss_final_chain:
    ss[2] = cid
    ss_final.append(ss)
    print(ss)




In [ ]:
#@title **Create custom secondary structure block adjacencies**

#@markdown **Click on cells to add or remove adjacencies** (symmetric adjacencies will automatically be added. Note: click on light blue cells to remove.)

#@markdown Only adjacencies to WRAP secondary structure blocks can be added. This step allows you to customize the scaffold topology you would like to generate with your WRAPs.

js_str = "function toggleCellContent(cell, row, col, skip, len, total) {\n\
  if (row != col && skip == 0) {\n\
    var symcell = document.getElementById(`cell_${col}_${row}_${skip}_${len}_${total}`);\n\
    if (symcell.style.backgroundColor == 'magenta') { return; }\n\
    google.colab.kernel.invokeFunction(\"toggle_callback\", [row, col], {});\n\
    var offdiagMap = { '0': ['1', 'lightblue'], '1': ['0', 'white'] };\n\
    [cell.textContent, cell.style.backgroundColor] = [symcell.textContent, symcell.style.backgroundColor] = offdiagMap[cell.textContent] || ['', ''];\n\
    var symm_num = parseInt(total/len)\n\
    for (var i = 1; i <= symm_num; i++) {\n\
      symrow = row+(i*len)\n\
      symcol = col+(i*len)\n\
      if (symrow <= total && symcol <= total) {\n\
        var symcell1 = document.getElementById(`cell_${symrow}_${symcol}_${skip}_${len}_${total}`);\n\
        var symcell2 = document.getElementById(`cell_${symcol}_${symrow}_${skip}_${len}_${total}`);\n\
        var offdiagMap = { '0': ['1', 'lightblue'], '1': ['0', 'white'] };\n\
        [symcell1.textContent, symcell1.style.backgroundColor] = [symcell2.textContent, symcell2.style.backgroundColor] = offdiagMap[symcell1.textContent] || ['', ''];\n\
        google.colab.kernel.invokeFunction(\"toggle_callback\", [symrow, symcol], {});\n\
      }\n\
      diffrow = symrow-total\n\
      diffcol = symcol-total\n\
      if (diffrow >= 0 && symcol <= total) {\n\
        var symcell1 = document.getElementById(`cell_${diffrow}_${symcol}_${skip}_${len}_${total}`);\n\
        var symcell2 = document.getElementById(`cell_${symcol}_${diffrow}_${skip}_${len}_${total}`);\n\
        var offdiagMap = { '0': ['1', 'magenta'], '1': ['0', 'white'] };\n\
        [symcell1.textContent, symcell1.style.backgroundColor] = [symcell2.textContent, symcell2.style.backgroundColor] = offdiagMap[symcell1.textContent] || ['', ''];\n\
        google.colab.kernel.invokeFunction(\"toggle_callback\", [diffrow, symcol], {});\n\
      }\n\
      if (diffcol >= 0 && symrow <= total) {\n\
        var symcell1 = document.getElementById(`cell_${symrow}_${diffcol}_${skip}_${len}_${total}`);\n\
        var symcell2 = document.getElementById(`cell_${diffcol}_${symrow}_${skip}_${len}_${total}`);\n\
        var offdiagMap = { '0': ['1', 'magenta'], '1': ['0', 'white'] };\n\
        [symcell1.textContent, symcell1.style.backgroundColor] = [symcell2.textContent, symcell2.style.backgroundColor] = offdiagMap[symcell1.textContent] || ['', ''];\n\
        google.colab.kernel.invokeFunction(\"toggle_callback\", [symrow, diffcol], {});\n\
      }\n\
      symrow = row-(i*len)\n\
      symcol = col-(i*len)\n\
      if (symrow >= 0 && symcol >= 0) {\n\
        var symcell1 = document.getElementById(`cell_${symrow}_${symcol}_${skip}_${len}_${total}`);\n\
        var symcell2 = document.getElementById(`cell_${symcol}_${symrow}_${skip}_${len}_${total}`);\n\
        var offdiagMap = { '0': ['1', 'lightblue'], '1': ['0', 'white'] };\n\
        [symcell1.textContent, symcell1.style.backgroundColor] = [symcell2.textContent, symcell2.style.backgroundColor] = offdiagMap[symcell1.textContent] || ['', ''];\n\
        google.colab.kernel.invokeFunction(\"toggle_callback\", [symrow, symcol], {});\n\
      }\n\
    }\n\
  }\n\
}\n\
function updateGridClickEvents() {\n\
  var cells = document.querySelectorAll('.grid-item');\n\
  cells.forEach(cell => {\n\
    cell.addEventListener('click', () => {\n\
      var [row, col, skip, len, total] = cell.id.split('_').slice(1).map(Number);\n\
      toggleCellContent(cell, row, col, skip, len, total);\n\
    });\n\
  });\n\
}\n\
updateGridClickEvents();\n\
"
css_str = ".grid-item {\n\
  display: flex;\n\
  align-items: center;\n\
  justify-content: center;\n\
  width: 30px;\n\
  height: 30px;\n\
  background-color: lightgray;\n\
  border: 2px solid black;\n\
  box-sizing: border-box;\n\
  color: black;\n\
}\n\
.text {\n\
    height: 25px;\n\
    width: 50px;\n\
    border: 2px solid lightgray;\n\
    box-sizing: border-box;\n\
}\n\
.pos {\n\
    display: flex;\n\
    align-items: center;\n\
    justify-content: center;\n\
}\n\
.button {\n\
    height: 25px;\n\
    border: 2px solid black;\n\
    margin: 2px 0px 2px 0px;\n\
    box-sizing: border-box;\n\
    color: black;\n\
}\n\
"

if not os.path.exists('blueprint.js'):
  nf = open("blueprint.js", "w")
  nf.write(js_str)
  nf.close()
if not os.path.exists('blueprint.css'):
  nf = open("blueprint.css", "w")
  nf.write(css_str)
  nf.close()

from IPython.display import display
import ipywidgets as widgets
import random, string, re
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files, output

custom_adj = ss_final

def get_adj_ss(adj, txt, buff=0, mask_contacts=False):
  # select non-zero elements
  idx = []
  for i in range(len(adj)):
    if txt[i] > 0:
      idx.append(i)

  L = (len(idx) + 1) * buff + sum(txt)
  full_adj = np.full((L,L),2)
  full_sse = np.full((L,),3)
  n = buff
  for i in idx:
    ss = {"H":0, "E":1, "L":2, "?":3}[adj[i][i]]
    full_sse[n:n+txt[i]] = ss
    m = buff
    for j in idx:
      k = str(adj[i][j])
      if i == j:
        val = {"H":0,"E":0,"L":0,"?":2}[k]
      else:
        if mask_contacts and k == "1": k = "?"
        val = {"0":0,"1":1,"?":2}[k]
      full_adj[n:n+txt[i],m:m+txt[j]] = val
      m += txt[j] + buff
    n += txt[i] + buff
  return {"adj":full_adj,"sse":full_sse}

def create_adj_matrix(custom_adj):
    # Filter out the elements that are 'H' or 'E'
    filtered = [item[0] for item in custom_adj if item[0] in ['H', 'E']]
    txt = [item[1] for item in custom_adj if item[0] in ['H', 'E']]
    filtered_t_or_w = [item[3] for item in custom_adj if item[0] in ['H', 'E']]
    # Get the number of 'H' and 'E' elements (size of the adj matrix)
    elements = len(filtered)

    # Initialize the adj matrix with 0s
    adj = [["0" for _ in range(elements)] for _ in range(elements)]
    adj_t_or_w = [["0" for _ in range(elements)] for _ in range(elements)]

    # Set the diagonal elements to the filtered 'H' and 'E' values
    for i in range(elements):
        adj[i][i] = filtered[i]

    # Set the diagonal elements to the filtered 't' or 'w' values
    for i in range(elements):
        adj_t_or_w[i][i] = filtered_t_or_w[i]

    return adj, adj_t_or_w, elements, txt

def insert_loops(adj, loop_pos, loop_char="L"):
    # Determine the size of the new matrix
    adj = np.array(adj)
    new_size = adj.shape[0] + len(loop_pos)
    # Create a new matrix filled with '0'
    new_matrix = np.full((new_size, new_size), '0', dtype='<U1')
    # Create an iterator for the old matrix indices
    old_idx = 0
    # Iterate through the new matrix and insert old matrix values or 'L'
    for i in range(new_size):
        if i in loop_pos:
            new_matrix[i][i] = loop_char  # Insert loop_char at the loop position
        else:
            # Copy the corresponding row and column from the old matrix
            for j in range(new_size):
                if j in loop_pos:
                    continue  # Skip columns where loop_char is inserted
                new_j = j - np.searchsorted(loop_pos, j)  # Adjust for loop positions
                new_matrix[i][j] = adj[old_idx][new_j]
            old_idx += 1
    return list(new_matrix)

class blueprint_gui:
    def _toggle_callback(self, row, col):
        if row == col:
            new_value = {"H":"E","E":"L","L":"?","?":"H"}[self.adj[row][col]]
            self.txt[row] = {"H": 19, "E": 5, "L": 3, "?": 0}[new_value]
            self.adj[row][col] = new_value
            for i in range(self.elements):
                if i != row:
                    if new_value == "?":
                        self.adj[row][i] = self.adj[i][col] = "?"
                    elif self.adj[i][i] != "?" and new_value in ["L","H"]:
                        self.adj[row][i] = self.adj[i][col] = '0'
        else:
            if self.adj[row][row] not in ["L","?"] and self.adj[col][col] not in ["L","?"]:
                new_value = {"0":"1","1":"?","?":"0"}[self.adj[row][col]]
                self.adj[row][col] = self.adj[col][row] = new_value

    def _create_html(self):
        # HTML for initial grid
        html_grid = f'<div class="pos"></div>'
        for row in range(self.elements):
          cid = int((row+self.elements_per_asymmetric_unit)/self.elements_per_asymmetric_unit)
          html_grid += f'<div class="pos">{cid}-{row+1}</div>'
        html_grid += f'<div class="pos"></div>'
        for row in range(self.elements):
            cid = int((row+self.elements_per_asymmetric_unit)/self.elements_per_asymmetric_unit)
            html_grid += f'<div class="pos">{cid}-{row+1}</div>'
            for col in range(self.elements):
                value = self.adj[row][col]
                t_or_w_value_row = self.adj_t_or_w[row][row]
                t_or_w_value_col = self.adj_t_or_w[col][col]
                skip = 0
                if t_or_w_value_row == 't' and t_or_w_value_col == 't':
                  skip = 1
                bgcolor = {"H":"red","E":"yellow","L":"lime","?":"lightgray","0":"white","1":"lightblue"}[value]
                if row != col and (self.adj[row][row] in ["?","L"] or self.adj[col][col] in ["?","L"]):
                    opacity = 0.1
                else:
                    opacity = 1.0
                html_grid += f'<div class="grid-item" id="cell_{row}_{col}_{skip}_{int(self.elements_per_asymmetric_unit)}_{int(self.elements)}" style="background-color:{bgcolor};opacity:{opacity}">{value}</div>'
            html_grid += f'<div class="text" type="number" id="cell_{row}">{self.txt[row]}</div>'

        self.html_code = f"""
        <style>
        {self._CSS}
        .grid-container {{
            display: grid;
            grid-template-columns: repeat({self.elements+2}, 30px);
            gap: 2px;
        }}
        </style>
        <script>{self._JS}</script>
        <div class="grid-container">{html_grid}</div>
        """

class RFdiff_gui(blueprint_gui):

    def __init__(self, elements=5, asymmetric_units=None, custom_adj=None, adj=None, txt=None, buff_length=0, name="test"):
        self.path = self.name = name
        self.input = widgets.Output()
        self.output = widgets.Output()
        self.buff_length = buff_length
        self.asymmetric_units = asymmetric_units
        small_button_style = widgets.Layout(width='30px', height='30px', border='2px solid black')
        button_style = widgets.Layout(width='84px', height='35px', border='2px solid black')
        self.buttons = {
                "buff_length": widgets.BoundedIntText(description='buff_length', value=self.buff_length, min=0, max=20),
                "reset":             widgets.Button(description='reset',         layout=button_style),
                "animate":         widgets.Button(description='animate',     layout=button_style),
                "freeze":            widgets.Button(description='freeze',        layout=button_style),
                "download":        widgets.Button(description='download',    layout=button_style),
                "color":             widgets.Dropdown(
                                                options=['SSE','pLDDT'],
                                                value='SSE',
                                                description='color',
                                                disabled=False)
        }

        self._plot = {"mode":"freeze","color":"SSE"}

        # prep inputs
        self.loops = []
        self.adj, self.adj_t_or_w, self.elements, self.txt = create_adj_matrix(custom_adj)
        self.elements_per_asymmetric_unit = self.elements/self.asymmetric_units

        output.register_callback("toggle_callback", self._toggle_callback)
        self._CSS = open("blueprint.css","r").read()
        self._JS = open("blueprint.js","r").read()

    def _redraw(self):
        with self.input:
            self._create_html()
            self.input.clear_output(wait=True)
            display(
                    widgets.VBox([
                    widgets.HTML(self.html_code),
                ])
            )

    def display_input(self):
        self._redraw()
        display(self.input)

    def _get_adj_ss(self, mask_contacts=False):
        # get unique path
        full = get_adj_ss(adj=self.adj,
                        txt=self.txt,
                        buff=self.buttons["buff_length"].value,
                        mask_contacts=mask_contacts)
        self._sse = full["sse"]
        self._adj = full["adj"]
        print(self._adj)


    def _get_adj_ss_withloops(self):
        if custom_adj is None:
            raise ValueError("Must be used with custom custom_adj, but it is none, pass one when defining the function")
        loop_pos = np.argwhere(np.array(custom_adj)[:, 0] == 'L').flatten()
        self.adj_wloop = insert_loops(self.adj, loop_pos)
        self.txt_wloop = np.array(custom_adj)[:, 1].astype(int)
        self.elements_wloop = len(self.adj_wloop)
        full = get_adj_ss(adj=self.adj_wloop,
                        txt=self.txt_wloop,
                        buff=self.buttons["buff_length"].value)
        self._sse_wloop = full["sse"]
        self._adj_wloop = full["adj"]
        self._elements_wloop = len(full["adj"])


    def plot_adj(self):
        self._get_adj_ss_withloops()
        """Plots a matrix with yellow for 1 and blue for 0."""
        plt.imshow(np.where(self._adj_wloop == 1, 1, 0), cmap='viridis')
        plt.show()

rfdiff = RFdiff_gui(asymmetric_units=asymmetric_units, custom_adj=ss_final)
rfdiff.display_input()



In [ ]:
#@title **Show SS elements adjacency contacts**
adj_element_contact = []
adj = rfdiff.adj
for element in rfdiff.adj:
    adj_element_contact.append([int(x) for x in np.argwhere(np.array(element) == '1').flatten()])
adj_element_contact

In [ ]:
#@title **Generate block_adjacency for diffusion**


import torch

rfdiff._get_adj_ss_withloops()
tensor_adj_matrix = torch.tensor(rfdiff._adj_wloop, dtype=torch.float32)
tensor_ss_strand = torch.tensor(rfdiff._sse_wloop, dtype=torch.float32)


if not os.path.isdir("adj"): os.mkdir("adj")
ss_strand_filepath = 'adj/adj_ss.pt'
adj_matrix_filepath = 'adj/adj_adj.pt'

torch.save(tensor_adj_matrix, adj_matrix_filepath)
torch.save(tensor_ss_strand, ss_strand_filepath)

plt.imshow(rfdiff._adj_wloop)

In [ ]:
#@title **Run scaffold-guided RFdiffusion with symmetry** (~10min to many hours)
%%time
num_designs = 5 #@param ["1","2","3","4","5","6","7","8","9","10"] {type:"raw"}

contigs_str = ""
for chain in chain_resnums:
  if nwraplen > 0:
    contigs_str += f"{nwraplen}/"
  contigs_str += f"{chain}{chain_resnums[chain][0]}-{chain_resnums[chain][-1]}/"
  if cwraplen > 0:
    contigs_str += f"{cwraplen}/"
  contigs_str += "0 "
contigs_str = contigs_str[0:-1]

# clear previous run if one exists
run_cmd("rm -rf diffusion_output")

cmd = f"RFdiffusion/scripts/run_inference.py "
cmd += f"--config-name=symmetry inference.symmetry='C{asymmetric_units}' "
cmd += f"inference.output_prefix=diffusion_output/diffusion inference.input_pdb=input.pdb "
cmd += f"inference.num_designs={num_designs} "
cmd += f"contigmap.contigs=['{contigs_str}'] "
cmd += f"inference.model_runner=NRBStyleSelfCond denoiser.noise_scale_ca=0.5 denoiser.noise_scale_frame=0.5 "
cmd += f"scaffoldguided.scaffoldguided=True scaffoldguided.scaffold_dir=adj diffuser.T=30"
os.environ["HYDRA_FULL_ERROR"] = '1'
print(cmd)
run_cmd(cmd)

rfd_mods = []
import glob
for rfd in glob.glob('diffusion_output/diffusion*.pdb'):
  rfd_mods.append(rfd)

In [ ]:
#@title **Select scaffold-guided RFdiffusion output for MPNN and AF3**
import py3Dmol
import ipywidgets as widgets
from ipywidgets import interact, Layout
from IPython.display import display, clear_output

#@markdown If the WRAPs do not look ideal, experiment with different configurations of SS blocks and adjacencies.

current_rfd = ""
dropdown = widgets.Dropdown(
  options=rfd_mods,
  description='backbones:',
  layout=Layout(width='50%', overflow='visible')
)
def on_dropdown_change(rfd):
  global current_rfd
  current_rfd = rfd
  clear_output(wait=True)
  print()
  print(current_rfd)
  view = py3Dmol.view(width=500, height=400)
  with open(rfd, "r") as f:
    pdb_data = f.read()
  view.addModel(pdb_data, 'pdb')
  view.setStyle({'cartoon': {'colorscheme': 'chain'}})
  view.zoomTo()
  view.show()

widgets.interact(on_dropdown_change, rfd=dropdown);



In [ ]:
#@title **Run tied (symmetric) Soluble ProteinMPNN**
%%time
if not os.path.exists(current_rfd):
  raise Exception("You must select an RFdiffused backbone.")
from pyrosetta.rosetta.core.pose import append_pose_to_pose
from pyrosetta.rosetta.core.kinematics import MoveMap
from pyrosetta.rosetta.protocols.minimization_packing import MinMover

#@markdown Select to minimize sidechains in MPNN PDB outputs (optional for aesthetics)
minimize = False #@param {type:"boolean"}
sequences_per_target = 5 #@param ["1","2","3","4","5","6","7","8","9","10","11","12","13","14","15","16","17","18","19","20" ] {type: "raw"}

import glob

# copy rfd backbone to run mpnn and af3 on
designsdir = "rfd_backbones"
if not os.path.isdir(designsdir):
  os.mkdir(designsdir)
os.system(f"cp {current_rfd} {designsdir}/")

chains_to_design = " ".join(chains.keys())

# target positions to fix sequence
chain_fixed_pos = []
for c in chain_resnums:
  fixed_pos = []
  pos = 0
  for resnum in chain_resnums[c]:
    pos += 1
    fixed_pos.append(f'{nwraplen + pos}')
  chain_fixed_pos.append(' '.join(fixed_pos))
fixed_positions = ', '.join(chain_fixed_pos)

path_for_parsed_chains=f"./{designsdir}/parsed_pdbs.jsonl"
path_for_tied_positions=f"./{designsdir}/tied_pdbs.jsonl"
path_for_designed_sequences=f"./{designsdir}_mpnn"
path_for_fixed_positions=f"./{designsdir}/fixed_pdbs.jsonl"

rfdbackbones = []
for outpdb in glob.glob(f"{designsdir}/*.pdb"):
  rfdbackbones.append(outpdb)
if len(rfdbackbones) == 0:
  raise Exception("There are no scaffold-guided RFDiffusion backbones to sequence design.")

run_cmd(f"python ProteinMPNN/helper_scripts/parse_multiple_chains.py --input_path={designsdir} --output_path={path_for_parsed_chains}")
run_cmd(f'python ProteinMPNN/helper_scripts/make_fixed_positions_dict.py --input_path={path_for_parsed_chains} --output_path={path_for_fixed_positions} --chain_list "{chains_to_design}" --position_list "{fixed_positions}"')
# Create tied positions jsonl file from homooligomer RFdiffusion outputs
run_cmd(f'python ProteinMPNN/helper_scripts/make_tied_positions_dict.py --input_path={path_for_parsed_chains} --output_path={path_for_tied_positions} --homooligomer 1')
run_cmd(f'python ProteinMPNN/protein_mpnn_run.py \
        --path_to_model_weights "./ProteinMPNN/soluble_model_weights/"\
        --jsonl_path {path_for_parsed_chains} \
        --tied_positions_jsonl {path_for_tied_positions} \
        --fixed_positions_jsonl {path_for_fixed_positions} \
        --out_folder {path_for_designed_sequences} \
        --num_seq_per_target {sequences_per_target} \
        --sampling_temp "0.2" \
        --batch_size 1')

alpha_1 = list("ARNDCQEGHILKMFPSTWYV-")
alpha_3 = ['ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS','ILE',
           'LEU','LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL','GAP']
aa_1_3 = {a:b for a,b in zip(alpha_1,alpha_3)}

def append_chain_to_pose(p1a,p2a,chain=1,new_chain=True):
  jumpadded = False
  for res in pyrosetta.rosetta.core.pose.get_chain_residues(p2a,chain):
    if not jumpadded:
      p1a.append_residue_by_jump(res, 1, "", "", new_chain)
      jumpadded = True
    else:
      p1a.append_residue_by_bond(res)
  return p1a

def thread_mpnn_seq( pose, binder_seq ):
  rsd_set = pose.residue_type_set_for_pose( pyrosetta.rosetta.core.chemical.FULL_ATOM_t )
  for resi, mut_to in enumerate( binder_seq ):
    resi += 1 # 1 indexing
    if pose.residue(resi).name().split(':')[-1] != 'disulfide':
      name3 = aa_1_3[ mut_to ]
      new_res = pyrosetta.rosetta.core.conformation.ResidueFactory.create_residue( rsd_set.name_map( name3 ) )
      pose.replace_residue( resi, new_res, True )
  return pose

import glob
seqs = []
for fasta in glob.glob(f"{path_for_designed_sequences}/seqs/*.fa"):
  input_pdb_name = ""
  with open(fasta) as f:
    scores = []
    for l in f:
      if l.startswith('>'):
        scores = l.split()
        if input_pdb_name == "": input_pdb_name = scores[0][1:-1]
      elif not scores[1].startswith("score="):
        seqs.append([input_pdb_name, scores, l.strip()])
for seq in seqs:
  idx = seq[1][1].split('=')[-1][0:-1]
  input_pdb = f"{designsdir}/{seq[0]}.pdb"
  #print(f'input {input_pdb}')
  pose = pyrosetta.pose_from_file(input_pdb)
  chainseqs = seq[2].split('/')
  singleseq = seq[2].replace('/','')
  pose = thread_mpnn_seq( pose, singleseq )
  chainlen = len(chainseqs[0])
  chains_pose = Pose()
  residue_indices = rosetta.utility.vector1_unsigned_long()
  for i in range(1,chainlen+1):
    residue_indices.append(i)
  pyrosetta.rosetta.core.pose.pdbslice(chains_pose, pose, residue_indices)
  for i in range(1,len(chainseqs)):
    start = chainlen*i+1
    residue_indices = rosetta.utility.vector1_unsigned_long()
    for j in range(start,start+chainlen):
      residue_indices.append(j)
    chainpose = Pose()
    pyrosetta.rosetta.core.pose.pdbslice(chainpose, pose, residue_indices)
    append_pose_to_pose(chains_pose, chainpose, True)

  if minimize:
    mm = MoveMap()
    mm.set_bb(False)  # Fix backbone
    mm.set_chi(True)  # Allow side chains to move
    mm.set_jump(False)
    scorefxn = create_score_function("ref2015")
    min_mover = MinMover()
    min_mover.movemap(mm)
    min_mover.score_function(scorefxn)
    print(f"minimize {path_for_designed_sequences}/{seq[0]}_{idx}.pdb")
    min_mover.apply(chains_pose)

  print(f"output {path_for_designed_sequences}/{seq[0]}_{idx}.pdb")
  chains_pose.dump_pdb(f"{path_for_designed_sequences}/{seq[0]}_{idx}.pdb")


In [ ]:
#@title **Download and install the AlphaFold3 git repo** (~5-10min)
%%time
if not os.path.isdir("alphafold3"):
  # get specific branch this was tested on in 20260204
  os.system("git clone https://github.com/google-deepmind/alphafold3.git; cd alphafold3; git fetch --all; git reset --hard ebfe70a")

if not os.path.isdir('alphafold3_venv'):
  # build the uv environment for running alphafold3
  run_cmd("export UV_COMPILE_BYTECODE=1;export UV_PROJECT_ENVIRONMENT=/content/alphafold3_venv; uv venv $UV_PROJECT_ENVIRONMENT;export PATH=\"/content/alphafold3_venv/bin:$PATH\"; cd alphafold3; uv sync --all-groups ; uv run build_data")


In [ ]:
#@title **Upload or mount drive for AlphaFold 3 model parameters.** (A very long time unless you use your Google Drive)
#@markdown **Copyright 2024 DeepMind Technologies Limited**
#@markdown
#@markdown AlphaFold 3 source code is licensed under CC BY-NC-SA 4.0. To view a copy of
#@markdown this license, visit https://creativecommons.org/licenses/by-nc-sa/4.0/
#@markdown
#@markdown To request access to the AlphaFold 3 model parameters, follow the process set
#@markdown out at https://github.com/google-deepmind/alphafold3. You may only use these
#@markdown if received directly from Google. Use is subject to terms of use available at
#@markdown https://github.com/google-deepmind/alphafold3/blob/main/WEIGHTS_TERMS_OF_USE.md

import os

#@markdown **We recommend first uploading the model parameters to your Goggle Drive and then mount the drive.**

#@markdown Select to mount your Google Drive (you will need to follow the popup instructions to authenticate)
mount_myDrive = True #@param {type:"boolean"}

#@markdown Enter the path to your AlphaFold 3 models directory in 'My Drive'
path_to_models_dir = "af3_models" #@param {type: "string"}

#@markdown The directory above should contain af3.bin, af3.bin.zst, and the params directory

af3_models_dir = ''
if mount_myDrive:
  mount_path = '/content/drive'
  if not os.path.exists(mount_path):
    from google.colab import drive
    drive.mount(mount_path)
  af3_models_dir = f"{mount_path}/MyDrive/{path_to_models_dir}"
  if os.path.isdir(af3_models_dir) and os.path.exists(f"{af3_models_dir}/params/params_model_1_multimer.npz") and os.path.exists(f"{af3_models_dir}/af3.bin"):
    print(f"Using AlphaFold 3 params from your Google Drive at {af3_models_dir}")
  else:
    raise Exception("Could not find models directory. Is your path correct?")
else:
  from google.colab import files
  if not os.path.isdir("af3_models"):
    os.mkdir("af3_models")
  if not os.path.isdir("af3_models/params"):
    os.mkdir("af3_models/params")
  uploads = files.upload()
  for upload_key in list(uploads.keys()):
    upload_name = upload_key.split('/')[-1].split()[0]
    if not os.path.exists(f"af3_models/af3.bin") and upload_name == "af3.bin":
      print(f"Upload upload_name into af3_models/{upload_name}")
      with open(f"af3_models/{upload_name}", "wb") as out: out.write(uploads[upload_key])
    elif not os.path.exists(f"af3_models/af3.bin.zst") and upload_name == "af3.bin.zst":
      print(f"Upload upload_name into af3_models/{upload_name}")
      with open(f"af3_models/{upload_name}", "wb") as out: out.write(uploads[upload_key])
    elif not os.path.exists(f"af3_models/params/{upload_name}"):
      print(f"Upload upload_name into af3_models/params/{upload_name}")
      with open(f"af3_models/params/{upload_name}", "wb") as out: out.write(uploads[upload_key])
  af3_models_dir = 'af3_models'

af3_database_dir = 'af3_database'
if not os.path.isdir(af3_database_dir):
  # create a placeholder database directory
  run_cmd(f"mkdir {af3_database_dir}; cd {af3_database_dir}; touch bfd-first_non_consensus_sequences.fasta mgy_clusters_2022_05.fa nt_rna_2023_02_23_clust_seq_id_90_cov_80_rep_seq.fasta pdb_seqres_2022_09_28.fasta rfam_14_9_clust_seq_id_90_cov_80_rep_seq.fasta rnacentral_active_seq_id_90_cov_80_linclust.fasta uniprot_all_2021_04.fa uniref90_2022_05.fa; mkdir mmcif_files")
print(af3_models_dir)
run_cmd(f"ls -lh {af3_models_dir}")

In [ ]:
#@title **Run AlphaFold 3 on the MPNN outputs** (5-10min per model, A100 GPU or Nvidia GPU compute capability 8.0+ required)
%%time

# Copyright 2024 DeepMind Technologies Limited
#
# AlphaFold 3 source code is licensed under CC BY-NC-SA 4.0. To view a copy of
# this license, visit https://creativecommons.org/licenses/by-nc-sa/4.0/
#
# To request access to the AlphaFold 3 model parameters, follow the process set
# out at https://github.com/google-deepmind/alphafold3. You may only use these
# if received directly from Google. Use is subject to terms of use available at
# https://github.com/google-deepmind/alphafold3/blob/main/WEIGHTS_TERMS_OF_USE.md

#@markdown This cell will run AlphaFold 3 without using MSAs from genetic or template searches.

#@markdown Alternatively you can download the MPNN sequences and submit them to the AlphaFold 3 server ending here in this notebook.

#@markdown The MPNN results are located in the rfd_backbones_mpnn folder.

#@markdown **This will not run on a T4 GPU**

import glob
import json

def read_pdb_atom(l):
  chain = l[20:22].strip()
  atype = l[11:17].strip()
  name3 = l[17:20].strip()
  resnum = int(l[22:26].strip())
  x = float(l[30:38])
  y = float(l[38:46])
  z = float(l[46:54])
  return chain, atype, name3, resnum, x, y, z

longer_names = {'ALA': 'A', 'ARG': 'R', 'ASN': 'N', 'ASP': 'D',
              'CYS': 'C', 'GLU': 'E', 'GLN': 'Q', 'GLY': 'G',
              'HIS': 'H', 'ILE': 'I', 'LEU': 'L', 'LYS': 'K',
              'MET': 'M', 'PHE': 'F', 'PRO': 'P', 'SER': 'S',
              'THR': 'T', 'TRP': 'W', 'TYR': 'Y', 'VAL': 'V' }

for mpnnout in glob.glob(f'{path_for_designed_sequences}/*.pdb'):
  in_name = mpnnout.split('/')[-1].split('.pdb')[0]
  chain_seqs = {}
  # get chains and single letter sequences
  with open(mpnnout) as f:
    for l in f:
      if l.startswith("ATOM"):
        chain, atype, name3, resnum, x, y, z = read_pdb_atom(l)
        if atype == 'CA':
          if not chain in chain_seqs:
            chain_seqs[chain] = longer_names[name3]
          else:
            chain_seqs[chain] += longer_names[name3]
  chains = []
  prevseq = ''
  for chain in chain_seqs:
    chains.append(chain)
    if prevseq and chain_seqs[chain] != prevseq:
      raise Exception(f"{mpnnout} is not a homo-oligomer.")
    prevseq = chain_seqs[chain]

  # Create AF3 input json file
  json_in = {
    "name": in_name,
    "sequences": [
      {
        "protein": {
          "id": chains,
          "sequence": prevseq,
          "unpairedMsa": "",
          "pairedMsa": "",
          "templates": ""
        }
      }
    ],
    "modelSeeds": [1],
    "dialect": "alphafold3",
    "version": 1
  }
  with open(f"{path_for_designed_sequences}/{in_name}.json", "w") as json_file:
    json.dump(json_in, json_file, indent=4)

# Run AF3 on each input json file
for json_in in glob.glob(f"{path_for_designed_sequences}/*.json"):
  name = json_in.split('/')[-1].split('.json')[0]
  json_prefix = json_in.split('.json')[0]
  print(name)
  cmd = f"export JAX_PLATFORMS=cuda; alphafold3_venv/bin/python "
  cmd += f"alphafold3/run_alphafold.py --json_path={json_in} "
  cmd+= f"--db_dir={af3_database_dir} --model_dir={af3_models_dir} "
  cmd+= f"--output_dir=af3_output > {json_prefix}.run_log 2>&1"
  print(cmd)
  run_cmd(cmd)


In [ ]:
#@title **Process AlphaFold 3 results**
%%time

import json
import pandas as pd
import glob
import os
import statistics
from Bio.PDB import MMCIFParser, PDBIO

import pyrosetta
from pyrosetta.rosetta import *
from pyrosetta.rosetta.core import *
pyrosetta.init(" -mute all ")

af3_mods = []
output_csv_path = f'af3_predictions.csv'
parsed_df = pd.DataFrame()
if os.path.exists(output_csv_path):
  parsed_df = pd.read_csv(output_csv_path, header=0)
else:
  parsed_data = []
  output_dir_list = glob.glob(f'af3_output/*')
  for output_dir in output_dir_list:
    for filename in os.listdir(output_dir):
      if filename.endswith('summary_confidences.json'):
        file_path = os.path.join(output_dir, filename)
        with open(file_path, 'r') as file:
          data = json.load(file)
          file_path2= file_path.split('summary_')[0] + file_path.split('summary_')[-1]
          with open(file_path2, 'r') as file2:
            data2 = json.load(file2)
          # Extract the required values
          iptm = data.get("iptm")
          ptm = data.get("ptm")
          # Extract the PAE values
          pae_values = data["chain_pair_pae_min"]
          plddt_total = statistics.mean(data2["atom_plddts"])
          design_name = output_dir.split('/')[-1]

          af3_model = f'{output_dir}/{design_name}_model.cif'
          mpnn_model = f'{path_for_designed_sequences}/{design_name}.pdb'

          af3_model_pdb = af3_model.split('.cif')[0]+'_af3.pdb'
          if not os.path.exists(af3_model_pdb):
            parser = MMCIFParser()
            structure = parser.get_structure(design_name, af3_model)
            io = PDBIO()
            io.set_structure(structure)
            io.save(af3_model_pdb)

          # superimpose mpnn_model to af3_model
          af3_pose = pose_from_file(af3_model_pdb)
          mpnn_pose = pose_from_file(mpnn_model)
          af3_chains = af3_pose.split_by_chain()
          mpnn_chains = mpnn_pose.split_by_chain()
          rmsd = scoring.CA_rmsd(af3_pose, mpnn_pose, 1, af3_pose.size())
          rmsd_subunit = scoring.CA_rmsd(af3_chains[1], mpnn_chains[1], 1, af3_chains[1].size())
          parsed_entry = {
            'design_name': design_name,
            'af3_model': af3_model_pdb,
            'mpnn_model': mpnn_model,
            'iptm': iptm,
            'ptm': ptm,
            'plddt': plddt_total,
            'rmsd': rmsd,
            'rmsd_subunit': rmsd_subunit
          }
          #print(f"{af3_model_pdb} iptm: {iptm:.2f} ptm: {ptm:.2f} plddt: {plddt_total:.2f} rmsd: {rmsd:.2f} rmsd_subunit: {rmsd_subunit:.2f}")
          # Append the parsed entry to the list
          parsed_data.append(parsed_entry)
          # add score info to af3 pdb
          has_scores = False
          fstr = ''
          with open(af3_model_pdb) as f:
            for l in f:
              if l.startswith("REMARK   af3_scores"):
                has_scores = True
                break
              else:
                fstr += l
          if not has_scores:
            nf = open(f"{af3_model_pdb}.tmp", 'w')
            nf.write(f"REMARK   af3_scores iptm: {iptm:.2f} ptm: {ptm:.2f} plddt: {plddt_total:.2f} rmsd: {rmsd:.2f} rmsd_subunit: {rmsd_subunit:.2f}\n")
            nf.write(fstr)
            nf.close()
            os.system(f"mv {af3_model_pdb}.tmp {af3_model_pdb}")
          af3_mods.append(af3_model_pdb)

  # Convert the list to a DataFrame
  parsed_df = pd.DataFrame(parsed_data)
  # Save the parsed DataFrame to a CSV file
  #parsed_df.to_csv(output_csv_path, index=False)

parsed_df

In [ ]:
#@title **Select AlphaFold 3 model to download**
import py3Dmol
import ipywidgets as widgets
from ipywidgets import interact, Layout
from IPython.display import display, clear_output

current_af3 = ""
af3_mods = { 'Select model to download': ''}
for i,r in parsed_df.iterrows():
  af3_mods[f"{r['af3_model']}  iptm: {r['iptm']:.2f} ptm: {r['ptm']:.2f} plddt: {r['plddt']:.2f} rmsd: {r['rmsd']:.2f} rmsd_subunit: {r['rmsd_subunit']:.2f}"] = r['af3_model']

dropdown = widgets.Dropdown(
  options=af3_mods,
  description='af2 wrap:',
  layout=Layout(width='50%', overflow='visible')
)
def on_dropdown_change(af3_mod):
  if os.path.exists(af3_mod):
    print(af3_mod)
    global current_af3
    current_af3 = af3_mod
    clear_output(wait=True)
    print()
    print(current_af3)
    view = py3Dmol.view(width=500, height=400)
    with open(af3_mod, "r") as f:
      pdb_data = f.read()
    view.addModel(pdb_data, 'pdb')
    view.setStyle({'cartoon': {'colorscheme': 'chain'}})
    view.zoomTo()
    view.show()

widgets.interact(on_dropdown_change, af3_mod=dropdown);

In [ ]:
#@title **Download selected AlphaFold 3 model**
from google.colab import files
files.download(current_af3)